# Load libraries

In [1]:
suppressMessages(library("data.table"))
suppressMessages(library("withr"))
suppressMessages(library("ggplot2"))
suppressMessages(library("farver"))
suppressMessages(library("labeling"))
suppressMessages(library("optparse"))
suppressMessages(library("dplyr"))
suppressMessages(library("backports"))
suppressMessages(library("broom"))
suppressMessages(library("rstudioapi"))
suppressMessages(library("tzdb"))
suppressMessages(library("svglite"))
suppressMessages(library("ggeasy"))
suppressMessages(library("tidyverse"))
suppressMessages(library("BiocGenerics"))
suppressMessages(library("S4Vectors"))
suppressMessages(library("IRanges"))
suppressMessages(library("GenomeInfoDb"))
suppressMessages(library("GenomicRanges"))
suppressMessages(library("Biobase"))
suppressMessages(library("AnnotationDbi"))
suppressMessages(library("GO.db"))
suppressMessages(library("org.Hs.eg.db"))
suppressMessages(library("ggrepel"))
suppressMessages(library("RColorBrewer"))
suppressMessages(library("cowplot"))
suppressMessages(library("Matrix"))
suppressMessages(library("rtracklayer"))
suppressMessages(library("Biostrings"))
suppressMessages(library("ggnewscale"))
suppressMessages(library("splitstackshape"))
suppressMessages(library("viridis"))
suppressMessages(library("ggsci"))
suppressMessages(library("plyr"))
suppressMessages(library("ggupset"))
suppressMessages(library("patchwork"))
suppressMessages(library("ggh4x"))
suppressMessages(library("ComplexHeatmap"))
suppressMessages(library("tidyr"))
suppressMessages(library("circlize"))
suppressMessages(library("grid"))
suppressMessages(library("gridExtra"))
suppressMessages(library("cluster"))
suppressMessages(library("clusterProfiler"))
suppressMessages(library("projectStyleR"))
library(lme4)
library(parallel)
library(UpSetR)



Attaching package: ‘lme4’


The following object is masked from ‘package:generics’:

    refit




## Load palettes

In [2]:
palettes_path <- system.file("palettes.yaml", package = "projectStyleR")
themes_path <- system.file("themes.yaml", package = "projectStyleR")
palettes <- yaml::read_yaml(palettes_path)
themes <- yaml::read_yaml(themes_path)


# GTF

In [13]:
gtf_path <- "/nfs/users/nfs_m/mt19/cardinal_analysis/ht/datasets/Flanders/trans_eQTLS_ERs/Homo_sapiens.GRCh38.110.gtf.gz"
gtf_cols <- c("seqname","source","feature","start","end","score","strand","frame","attribute")
gtf <- fread(gtf_path, sep = "\t", header = FALSE, col.names = gtf_cols, skip = "#")

genes_gtf <- gtf[feature == "gene"]
genes_gtf[, gene_id   := sub('.*gene_id "([^"]+)".*', "\\1", attribute)]
genes_gtf[, gene_name := sub('.*gene_name "([^"]+)".*', "\\1", attribute)]
ens2sym <- unique(genes_gtf[, .(gene_id, gene_name)])
stopifnot(!any(duplicated(ens2sym$gene_id)))  # one symbol per Ensembl ID, before trusting the merge



Rows with unmapped Source_symbol: 0 
Rows with unmapped Target_symbol: 0 


Source_module,Source_gene,Source_symbol,program,Source_cell,Target_cell,Target_gene,Target_symbol,edge_role,edge_resource,GWAS_colocalized,resource_base,direction
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD16,NK_CD16,ENSG00000196781,TLE1,Source_TF->Target,CollecTRI (>=2 CollecTRI sub-resources),FALSE,CollecTRI,forward (Source_gene = regulator/ligand/peptidase)
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD56bright,NK_CD56bright,ENSG00000196781,TLE1,Source_TF->Target,CollecTRI (>=2 CollecTRI sub-resources),FALSE,CollecTRI,forward (Source_gene = regulator/ligand/peptidase)
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD16,NK_CD16,ENSG00000197540,GZMM,Target_Peptidase->Source_Substrate,MEROPS (cleavage_type == physiological),FALSE,MEROPS,reverse (Source_gene = regulated/receptor/substrate)
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD56bright,NK_CD56bright,ENSG00000197540,GZMM,Target_Peptidase->Source_Substrate,MEROPS (cleavage_type == physiological),FALSE,MEROPS,reverse (Source_gene = regulated/receptor/substrate)


# Read in master files

In [3]:
suppressMessages(library("data.table"))

master_dir <- "/nfs/team151/mt19/overhaul_classification_factors_with_programs/"
whole_eqtl_annotated <- readRDS(paste0(master_dir, "whole_eqtl_annotated.rds"))
cat("whole_eqtl_annotated:", nrow(whole_eqtl_annotated), "rows\n")

whole_eqtl_annotated: 305496 rows


In [22]:
# Does HHEX have ANY SCENIC edge anywhere in the whole dataset, not just this factor?
hhex_scenic_check <- whole_eqtl_annotated[
  Source_gene == "ENSG00000152804" &
  !is.na(edge_role) &
  grepl("SCENIC", edge_resource, fixed = TRUE)
]
cat("HHEX rows with any SCENIC-tagged edge_resource, anywhere in the dataset:", nrow(hhex_scenic_check), "\n")

# And specifically: does THIS network have any SCENIC edges at all, even ones
# that might not have made it into unit_annotated for some other reason?
hhex_scenic_this_factor <- unit_all[!is.na(edge_role) & grepl("SCENIC", edge_resource, fixed = TRUE)]
cat("SCENIC-tagged rows within M_17602/D_Factor165 specifically:", nrow(hhex_scenic_this_factor), "\n")

HHEX rows with any SCENIC-tagged edge_resource, anywhere in the dataset: 0 
SCENIC-tagged rows within M_17602/D_Factor165 specifically: 0 


In [23]:
if (!exists("collectri_strict")) collectri_strict <- fread("/nfs/team151/mt19/annotation_preprocessed/collectri_strict.tsv")
if (!exists("scenic_activator"))  scenic_activator  <- fread("/nfs/team151/mt19/annotation_preprocessed/scenic_regulon_activator_same_dataset.tsv")
if (!exists("scenic_repressor"))  scenic_repressor  <- fread("/nfs/team151/mt19/annotation_preprocessed/scenic_regulon_repressor_same_dataset.tsv")

tf_target_pairs <- unique(rbind(
  collectri_strict[, .(TF_gene = partner1_ensembl_gene_id, target_gene = partner2_ensembl_gene_id,
                        resource = "CollecTRI", sign = NA_character_)],
  scenic_activator[, .(TF_gene = partner1_ensembl_gene_id, target_gene = partner2_ensembl_gene_id,
                        resource = "SCENIC_regulon_activator_same_dataset", sign = "Activator")],
  scenic_repressor[, .(TF_gene = partner1_ensembl_gene_id, target_gene = partner2_ensembl_gene_id,
                        resource = "SCENIC_regulon_repressor_same_dataset", sign = "Repressor")]
))
cat("Combined TF->target pairs available for cascade detection:", nrow(tf_target_pairs), "\n")

Combined TF->target pairs available for cascade detection: 178232 


# Cell 3 — Define the exact network to export

In [4]:
target_module  <- "M_17602"
target_program <- "D_Factor165"
target_gene    <- unique(whole_eqtl_annotated[Source_module == target_module & program == target_program]$Source_gene)
stopifnot(length(target_gene) == 1)
cat("Source_gene resolved to:", target_gene, "\n")

unit_all <- whole_eqtl_annotated[
  Source_module == target_module & program == target_program & Source_gene == target_gene
]
cat("Rows for this network (all Source_cell/Target_cell contexts):", nrow(unit_all), "\n")
cat("Source_cell(s) present:", paste(sort(unique(unit_all$Source_cell)), collapse = ", "), "\n")
cat("Target_cell(s) present:", paste(sort(unique(unit_all$Target_cell)), collapse = ", "), "\n")
stopifnot(nrow(unit_all) > 0)

# Sanity check against Source_symbol, since we already know this factor from earlier work
cat("Source_symbol:", unique(unit_all$Source_symbol), "(expect HHEX)\n")
stopifnot(unique(unit_all$Source_symbol) == "HHEX")

Source_gene resolved to: ENSG00000152804 
Rows for this network (all Source_cell/Target_cell contexts): 308 
Source_cell(s) present: NK_CD16, NK_CD56bright 
Target_cell(s) present: NK_CD16, NK_CD56bright 
Source_symbol: (expect HHEX)


# Cell 4 — Restrict to annotated edges only, and confirm the count

In [15]:
unit_annotated <- unit_all[!is.na(edge_role)]
cat("Annotated edges:", nrow(unit_annotated), "/", nrow(unit_all), "total edges in this network\n")

Annotated edges: 4 / 308 total edges in this network


# Cell 5 — Expand the ;-separated edge_role/edge_resource

In [16]:
role_split     <- strsplit(unit_annotated$edge_role, ";")
resource_split <- strsplit(unit_annotated$edge_resource, ";")
stopifnot(all(lengths(role_split) == lengths(resource_split)))  # confirms lockstep pairing before unlisting

n_per_row <- lengths(role_split)

edges_long <- data.table(
  Source_module = rep(unit_annotated$Source_module, n_per_row),
  Source_gene   = rep(unit_annotated$Source_gene, n_per_row),
  Source_symbol = rep(unit_annotated$Source_symbol, n_per_row),
  program       = rep(unit_annotated$program, n_per_row),
  Source_cell   = rep(unit_annotated$Source_cell, n_per_row),
  Target_cell   = rep(unit_annotated$Target_cell, n_per_row),
  Target_gene   = rep(unit_annotated$Target_gene, n_per_row),
  Target_symbol = rep(unit_annotated$Target_symbol, n_per_row),
  edge_role     = unlist(role_split),
  edge_resource = unlist(resource_split),
  GWAS_colocalized     = rep(unit_annotated$GWAS_colocalized, n_per_row),
  MR_effect            = rep(unit_annotated$MR_effect, n_per_row),
  most_likely_snp      = rep(unit_annotated$most_likely_snp, n_per_row),
  most_likely_snp_beta = rep(unit_annotated$most_likely_snp_beta, n_per_row)
)

cat("Expanded rows (one per edge x annotation label):", nrow(edges_long), "\n")

Expanded rows (one per edge x annotation label): 4 


# Cell 6 — Add a readable resource tag and direction, for the collaborator's benefit

In [17]:
edges_long[, resource_base := fcase(
  grepl("SCENIC_regulon_activator_same_dataset", edge_resource, fixed = TRUE), "SCENIC_activator",
  grepl("SCENIC_regulon_repressor_same_dataset", edge_resource, fixed = TRUE), "SCENIC_repressor",
  grepl("CollecTRI", edge_resource, fixed = TRUE), "CollecTRI",
  grepl("POSTAR3", edge_resource, fixed = TRUE), "POSTAR3_RBP",
  grepl("Liana", edge_resource, fixed = TRUE), "Liana_LR",
  grepl("MEROPS", edge_resource, fixed = TRUE), "MEROPS",
  grepl("STRING_experimental", edge_resource, fixed = TRUE), "STRING_PPI",
  default = NA_character_
)]
edges_long[, direction := fcase(
  grepl("^Source_", edge_role), "forward (Source_gene = regulator/ligand/peptidase)",
  grepl("^Target_", edge_role), "reverse (Source_gene = regulated/receptor/substrate)",
  grepl("^PPI", edge_role), "undirected",
  default = NA_character_
)]

stopifnot(sum(is.na(edges_long$resource_base)) == 0)  # every row should match one of the 7 resources
print(edges_long[, .N, by = .(resource_base, direction)])

   resource_base                                            direction     N
          <char>                                               <char> <int>
1:     CollecTRI   forward (Source_gene = regulator/ligand/peptidase)     2
2:        MEROPS reverse (Source_gene = regulated/receptor/substrate)     2


In [18]:
head(edges_long)

Source_module,Source_gene,program,Source_cell,Target_cell,Target_gene,edge_role,edge_resource,GWAS_colocalized,resource_base,direction
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>
M_17602,ENSG00000152804,D_Factor165,NK_CD16,NK_CD16,ENSG00000196781,Source_TF->Target,CollecTRI (>=2 CollecTRI sub-resources),FALSE,CollecTRI,forward (Source_gene = regulator/ligand/peptidase)
M_17602,ENSG00000152804,D_Factor165,NK_CD56bright,NK_CD56bright,ENSG00000196781,Source_TF->Target,CollecTRI (>=2 CollecTRI sub-resources),FALSE,CollecTRI,forward (Source_gene = regulator/ligand/peptidase)
M_17602,ENSG00000152804,D_Factor165,NK_CD16,NK_CD16,ENSG00000197540,Target_Peptidase->Source_Substrate,MEROPS (cleavage_type == physiological),FALSE,MEROPS,reverse (Source_gene = regulated/receptor/substrate)
M_17602,ENSG00000152804,D_Factor165,NK_CD56bright,NK_CD56bright,ENSG00000197540,Target_Peptidase->Source_Substrate,MEROPS (cleavage_type == physiological),FALSE,MEROPS,reverse (Source_gene = regulated/receptor/substrate)


In [19]:
"Source_symbol" %in% names(whole_eqtl_annotated)
"Target_symbol" %in% names(whole_eqtl_annotated)

[1] FALSE

[1] FALSE

In [20]:
# Source_symbol: constant across the whole factor, but merge properly rather
# than assume - safer than a bare unique() scalar if something's off upstream
edges_long <- merge(edges_long, ens2sym, by.x = "Source_gene", by.y = "gene_id", all.x = TRUE)
setnames(edges_long, "gene_name", "Source_symbol")

edges_long <- merge(edges_long, ens2sym, by.x = "Target_gene", by.y = "gene_id", all.x = TRUE)
setnames(edges_long, "gene_name", "Target_symbol")

# Confirm nothing came back unmapped before trusting the export
n_missing_source <- sum(is.na(edges_long$Source_symbol))
n_missing_target <- sum(is.na(edges_long$Target_symbol))
cat("Rows with unmapped Source_symbol:", n_missing_source, "\n")
cat("Rows with unmapped Target_symbol:", n_missing_target, "\n")

setcolorder(edges_long, c("Source_module", "Source_gene", "Source_symbol", "program",
                           "Source_cell", "Target_cell", "Target_gene", "Target_symbol",
                           "edge_role", "edge_resource"))
head(edges_long)

Rows with unmapped Source_symbol: 0 
Rows with unmapped Target_symbol: 0 


Source_module,Source_gene,Source_symbol,program,Source_cell,Target_cell,Target_gene,Target_symbol,edge_role,edge_resource,GWAS_colocalized,resource_base,direction
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD16,NK_CD16,ENSG00000196781,TLE1,Source_TF->Target,CollecTRI (>=2 CollecTRI sub-resources),FALSE,CollecTRI,forward (Source_gene = regulator/ligand/peptidase)
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD56bright,NK_CD56bright,ENSG00000196781,TLE1,Source_TF->Target,CollecTRI (>=2 CollecTRI sub-resources),FALSE,CollecTRI,forward (Source_gene = regulator/ligand/peptidase)
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD16,NK_CD16,ENSG00000197540,GZMM,Target_Peptidase->Source_Substrate,MEROPS (cleavage_type == physiological),FALSE,MEROPS,reverse (Source_gene = regulated/receptor/substrate)
M_17602,ENSG00000152804,HHEX,D_Factor165,NK_CD56bright,NK_CD56bright,ENSG00000197540,GZMM,Target_Peptidase->Source_Substrate,MEROPS (cleavage_type == physiological),FALSE,MEROPS,reverse (Source_gene = regulated/receptor/substrate)


In [21]:
dim(edges_long)

[1]  4 13

# Cascades

## Load the combined TF-target table (CollecTRI + both SCENIC regulons), if not already in session

In [24]:
if (!exists("collectri_strict")) collectri_strict <- fread("/nfs/team151/mt19/annotation_preprocessed/collectri_strict.tsv")
if (!exists("scenic_activator"))  scenic_activator  <- fread("/nfs/team151/mt19/annotation_preprocessed/scenic_regulon_activator_same_dataset.tsv")
if (!exists("scenic_repressor"))  scenic_repressor  <- fread("/nfs/team151/mt19/annotation_preprocessed/scenic_regulon_repressor_same_dataset.tsv")

tf_target_pairs <- unique(rbind(
  collectri_strict[, .(TF_gene = partner1_ensembl_gene_id, target_gene = partner2_ensembl_gene_id,
                        resource = "CollecTRI", sign = NA_character_)],
  scenic_activator[, .(TF_gene = partner1_ensembl_gene_id, target_gene = partner2_ensembl_gene_id,
                        resource = "SCENIC_regulon_activator_same_dataset", sign = "Activator")],
  scenic_repressor[, .(TF_gene = partner1_ensembl_gene_id, target_gene = partner2_ensembl_gene_id,
                        resource = "SCENIC_regulon_repressor_same_dataset", sign = "Repressor")]
))
cat("Combined TF->target pairs available for cascade detection:", nrow(tf_target_pairs), "\n")

Combined TF->target pairs available for cascade detection: 178232 


## Cell 9 — Detect cascade edges

In [25]:
cascade_edges_list <- list()

sub_units <- unique(unit_all[, .(Source_cell, Target_cell)])
for (i in seq_len(nrow(sub_units))) {
  sc <- sub_units$Source_cell[i]; tc <- sub_units$Target_cell[i]
  sub_data <- unit_all[Source_cell == sc & Target_cell == tc]

  target_genes_this_subunit <- unique(sub_data$Target_gene)
  target_genes_this_subunit <- target_genes_this_subunit[target_genes_this_subunit != target_gene]  # exclude the hub itself

  candidate_pairs <- tf_target_pairs[
    TF_gene %in% target_genes_this_subunit & target_gene %in% target_genes_this_subunit
  ]

  if (nrow(candidate_pairs) > 0) {
    candidate_pairs[, `:=`(Source_module = target_module, Source_gene = target_gene, program = target_program,
                            Source_cell = sc, Target_cell = tc)]
    cascade_edges_list[[length(cascade_edges_list) + 1]] <- candidate_pairs
  }
}

cascade_edges <- if (length(cascade_edges_list) > 0) rbindlist(cascade_edges_list) else
  data.table(TF_gene = character(0), target_gene = character(0), resource = character(0), sign = character(0),
             Source_module = character(0), Source_gene = character(0), program = character(0),
             Source_cell = character(0), Target_cell = character(0))

cat("Cascade edges found in this network:", nrow(cascade_edges), "\n")
print(cascade_edges[, .N, by = resource])

Cascade edges found in this network: 134 
                                resource     N
                                  <char> <int>
1:                             CollecTRI     6
2: SCENIC_regulon_activator_same_dataset   112
3: SCENIC_regulon_repressor_same_dataset    16


In [26]:
cascade_edges <- merge(cascade_edges, ens2sym, by.x = "TF_gene", by.y = "gene_id", all.x = TRUE)
setnames(cascade_edges, "gene_name", "TF_symbol")
cascade_edges <- merge(cascade_edges, ens2sym, by.x = "target_gene", by.y = "gene_id", all.x = TRUE)
setnames(cascade_edges, "gene_name", "target_symbol")

setcolorder(cascade_edges, c("Source_module", "Source_gene", "program", "Source_cell", "Target_cell",
                              "TF_gene", "TF_symbol", "target_gene", "target_symbol", "resource", "sign"))
print(cascade_edges)

Key: <target_gene>
     Source_module     Source_gene     program   Source_cell   Target_cell
            <char>          <char>      <char>        <char>        <char>
  1:       M_17602 ENSG00000034677 D_Factor165       NK_CD16       NK_CD16
  2:       M_17602 ENSG00000034677 D_Factor165 NK_CD56bright NK_CD56bright
  3:       M_17602 ENSG00000035403 D_Factor165       NK_CD16       NK_CD16
  4:       M_17602 ENSG00000035403 D_Factor165 NK_CD56bright NK_CD56bright
  5:       M_17602 ENSG00000054654 D_Factor165       NK_CD16       NK_CD16
 ---                                                                      
130:       M_17602 ENSG00000243927 D_Factor165 NK_CD56bright NK_CD56bright
131:       M_17602 ENSG00000256128 D_Factor165       NK_CD16       NK_CD16
132:       M_17602 ENSG00000256128 D_Factor165 NK_CD56bright NK_CD56bright
133:       M_17602 ENSG00000277972 D_Factor165       NK_CD16       NK_CD16
134:       M_17602 ENSG00000277972 D_Factor165 NK_CD56bright NK_CD56bright
      

# Export, both TSV and CSV

In [30]:
source_symbol<-unique(edges_long$Source_symbol)

In [31]:
source_symbol

[1] "HHEX"

In [32]:
export_dir <- paste0(master_dir, "collaborator_exports/")
dir.create(export_dir, recursive = TRUE, showWarnings = FALSE)

out_prefix <- paste0(target_module, "_", source_symbol, "_", target_program)

fwrite(edges_long, paste0(export_dir, out_prefix, "_annotated_edges.tsv"), sep = "\t")
fwrite(edges_long, paste0(export_dir, out_prefix, "_annotated_edges.csv"), sep = ",")

fwrite(cascade_edges, paste0(export_dir, out_prefix, "_cascade_edges.tsv"), sep = "\t")
fwrite(cascade_edges, paste0(export_dir, out_prefix, "_cascade_edges.csv"), sep = ",")
cat("Saved cascade edge table:", paste0(export_dir, out_prefix, "_cascade_edges.tsv"), "\n")

cat("Saved:\n")
cat(" ", paste0(export_dir, out_prefix, "_annotated_edges.tsv"), "\n")
cat(" ", paste0(export_dir, out_prefix, "_annotated_edges.csv"), "\n")

Saved cascade edge table: /nfs/team151/mt19/overhaul_classification_factors_with_programs/collaborator_exports/M_17602_HHEX_D_Factor165_cascade_edges.tsv 
Saved:
  /nfs/team151/mt19/overhaul_classification_factors_with_programs/collaborator_exports/M_17602_HHEX_D_Factor165_annotated_edges.tsv 
  /nfs/team151/mt19/overhaul_classification_factors_with_programs/collaborator_exports/M_17602_HHEX_D_Factor165_annotated_edges.csv 
